In [ ]:
from dotenv import load_dotenv # type: ignore
import os 
import requests
import json
import SRC
from SRC.helpFolder import get_coordinates, append_to_json_file
from SRC.helpFolder.helpersEtractPMS import fetch_and_filter_reservations_by_date_chunked

metaData = SRC.load_model_metadata() 

### Extract the data based on the **dates** 

In [4]:
metaData['PMSInformation']['start_date_range'], metaData['PMSInformation']['end_date_range']

('2024-05-01', '2025-08-30')

In [5]:
reservations_par_date, dates_list, summary_stats = fetch_and_filter_reservations_by_date_chunked(
    start_date_str=metaData['PMSInformation']['start_date_range'],
    end_date_str=metaData['PMSInformation']['end_date_range'],
    api_templates=metaData['PMSInformation']['api_templates'],
)

Starting chunked data fetch for range: 2024-05-01 to 2025-08-30
Chunk size: 5 days

--- Processing Chunk 1: 2024-05-01 to 2024-05-05 ---
Sending requests to 3 APIs for chunk 1
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-05-01&to=2024-05-05&group=1&resastatus=2
Chunk 1: Fetched 212 reservations

--- Processing Chunk 2: 2024-05-06 to 2024-05-10 ---
Sending requests to 3 APIs for chunk 2
Failed to fetch data from https://pmsvaleriaapi.fractalstay.com/api/reservations?&from=2024-05-06&to=2024-05-10&group=1&resastatus=2
Chunk 2: Fetched 427 reservations

--- Processing Chunk 3: 2024-05-11 to 2024-05-15 ---
Sending requests to 3 APIs for chunk 3
Chunk 3: Fetched 575 reservations

--- Processing Chunk 4: 2024-05-16 to 2024-05-20 ---
Sending requests to 3 APIs for chunk 4
Chunk 4: Fetched 556 reservations

--- Processing Chunk 5: 2024-05-21 to 2024-05-25 ---
Sending requests to 3 APIs for chunk 5
Failed to fetch data from https://pmsvaleriaapi.fr

In [6]:
def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")

In [7]:
append_to_json_file(
    reservations_par_date,
    '../Data/PMSEtractedData/reservations_par_date.json'
)

Data saved to ../Data/PMSEtractedData/reservations_par_date.json


In [8]:
summary_stats

{'total_chunks_processed': 98,
 'total_reservations_fetched': 59299,
 'total_reservations_filtered': 58126,
 'total_removed_reservations': 1173,
 'date_range': {'start': '2024-05-01', 'end': '2025-08-30', 'total_days': 487},
 'chunk_details': [{'chunk': 1,
   'start': '2024-05-01',
   'end': '2024-05-05',
   'reservations': 212},
  {'chunk': 2,
   'start': '2024-05-06',
   'end': '2024-05-10',
   'reservations': 427},
  {'chunk': 3,
   'start': '2024-05-11',
   'end': '2024-05-15',
   'reservations': 575},
  {'chunk': 4,
   'start': '2024-05-16',
   'end': '2024-05-20',
   'reservations': 556},
  {'chunk': 5,
   'start': '2024-05-21',
   'end': '2024-05-25',
   'reservations': 554},
  {'chunk': 6,
   'start': '2024-05-26',
   'end': '2024-05-30',
   'reservations': 589},
  {'chunk': 7,
   'start': '2024-05-31',
   'end': '2024-06-04',
   'reservations': 605},
  {'chunk': 8,
   'start': '2024-06-05',
   'end': '2024-06-09',
   'reservations': 621},
  {'chunk': 9,
   'start': '2024-06-10

In [9]:
from collections import defaultdict

def classify_by_segment_per_date(data_by_date):
    """
    Takes a dict with dates as keys and list of reservations as values,
    returns a nested dict where reservations are grouped by customerSegment per date.
    """
    result = {}

    for date, reservations in data_by_date.items():

        segment_groups = defaultdict(list)
        
        for res in reservations:
            segment = res.get("customerSegment", "UNKNOWN")
            segment_groups[segment].append(res)
        
        result[date] = dict(segment_groups)

    return result

In [10]:
classified = classify_by_segment_per_date(reservations_par_date)
for date, segments in classified.items():
    print(f"\nDate: {date}")
    for segment, res_list in segments.items():
        print(f"  Segment: {segment} | Count: {len(res_list)}")


Date: 2024-05-01
  Segment: TO | Count: 5
  Segment: AVM | Count: 1
  Segment: B2B | Count: 2

Date: 2024-05-02
  Segment: B2B | Count: 9
  Segment: None | Count: 9
  Segment: TO | Count: 14
  Segment: AVM | Count: 2
  Segment: Direct | Count: 1
  Segment: Onlines(OTA) | Count: 7

Date: 2024-05-03
  Segment: TO | Count: 51
  Segment: None | Count: 10
  Segment: AVM | Count: 7
  Segment: Onlines(OTA) | Count: 14
  Segment: B2B | Count: 10
  Segment: Direct | Count: 1

Date: 2024-05-04
  Segment: None | Count: 16
  Segment: TO | Count: 204
  Segment: Onlines(OTA) | Count: 28
  Segment: B2B | Count: 12
  Segment: AVM | Count: 47

Date: 2024-05-05
  Segment: Onlines(OTA) | Count: 41
  Segment: TO | Count: 350
  Segment: None | Count: 8
  Segment: B2B | Count: 16
  Segment: AVM | Count: 41

Date: 2024-05-06
  Segment: Onlines(OTA) | Count: 39
  Segment: TO | Count: 374
  Segment: None | Count: 20
  Segment: B2B | Count: 21
  Segment: AVM | Count: 48

Date: 2024-05-07
  Segment: Onlines(OTA

In [ ]:
from collections import defaultdict

def classify_by_segment_and_count(data_by_date):
    """
    Classifies reservations by customerSegment per date,
    and returns total counts per date.
    """
    classified = {}
    counts = {}

    for date, reservations in data_by_date.items():
        segment_groups = defaultdict(list)

        for res in reservations:
            segment = res.get("customerSegment", "UNKNOWN")
            segment_groups[segment].append(res)

        classified[date] = dict(segment_groups)

        counts[date] = {segment: len(res_list) for segment, res_list in segment_groups.items()}

    return classified, counts

In [18]:
classified_by_segment, counts_per_day = classify_by_segment_and_count(reservations_par_date)

In [19]:
counts_per_day

{'2024-05-01': {'TO': 5, 'AVM': 1, 'B2B': 2},
 '2024-05-02': {'B2B': 9,
  None: 9,
  'TO': 14,
  'AVM': 2,
  'Direct': 1,
  'Onlines(OTA)': 7},
 '2024-05-03': {'TO': 51,
  None: 10,
  'AVM': 7,
  'Onlines(OTA)': 14,
  'B2B': 10,
  'Direct': 1},
 '2024-05-04': {None: 16, 'TO': 204, 'Onlines(OTA)': 28, 'B2B': 12, 'AVM': 47},
 '2024-05-05': {'Onlines(OTA)': 41, 'TO': 350, None: 8, 'B2B': 16, 'AVM': 41},
 '2024-05-06': {'Onlines(OTA)': 39, 'TO': 374, None: 20, 'B2B': 21, 'AVM': 48},
 '2024-05-07': {'Onlines(OTA)': 44, 'TO': 400, None: 45, 'AVM': 19, 'B2B': 16},
 '2024-05-08': {'Onlines(OTA)': 52, 'TO': 410, 'AVM': 29, 'B2B': 30, None: 74},
 '2024-05-09': {'TO': 441,
  'Onlines(OTA)': 66,
  'AVM': 45,
  'B2B': 36,
  None: 111},
 '2024-05-10': {'TO': 485,
  'Onlines(OTA)': 73,
  'AVM': 89,
  None: 138,
  'B2B': 40},
 '2024-05-11': {'TO': 393, 'Onlines(OTA)': 61, None: 88, 'B2B': 39, 'AVM': 78},
 '2024-05-12': {'TO': 338, None: 44, 'B2B': 37, 'Onlines(OTA)': 50, 'AVM': 6},
 '2024-05-13': {'TO